# Hospital Patient Care Analytics Pipeline
## Kaggle Healthcare Analytics – Patient Flow Dataset

**Objective:** Build a hospital patient-care analytics workflow covering data ingestion, cleaning, validation, transformation, exploratory analytics, waiting-time analysis, admission analysis, patient satisfaction, and dashboard-ready outputs.

> **Dataset limitation:** This dataset contains patient flow, admission, demographic, satisfaction, and waiting-time fields. It does not contain laboratory reports, wearable readings, doctor notes, pharmacy data, or a clinical high-risk label. Therefore, those components are documented as future extensions rather than fabricated in this analysis.


In [ ]:
# 1. Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

DATA_PATH = 'healthcare_analytics_patient_flow_data.csv'
df = pd.read_csv(DATA_PATH)

print('Dataset loaded successfully')
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
df.head()


In [ ]:
# 2. Dataset inspection
print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes.to_frame('Data Type'))

print('\nFirst five records:')
display(df.head())

print('\nStatistical summary:')
display(df.describe(include='all').T)


In [ ]:
# 3. Data quality checks
print('Missing values:')
display(df.isnull().sum().to_frame('Missing Count'))

print('Duplicate rows:', df.duplicated().sum())

print('Unique patient IDs:', df['Patient Id'].nunique())
print('Duplicate patient IDs:', df['Patient Id'].duplicated().sum())

print('\nAdmission flag values:')
display(df['Patient Admission Flag'].value_counts(dropna=False))

print('\nDepartment referral values:')
display(df['Department Referral'].value_counts(dropna=False))


In [ ]:
# 4. Data cleaning and transformation
clean_df = df.copy()

# Standardize column names
clean_df.columns = (
    clean_df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

# Parse date and time fields
clean_df['patient_admission_date'] = pd.to_datetime(
    clean_df['patient_admission_date'], errors='coerce'
)
clean_df['patient_admission_time'] = pd.to_datetime(
    clean_df['patient_admission_time'], errors='coerce'
).dt.time

# Handle missing categorical values
for col in ['department_referral', 'patient_gender', 'patient_race']:
    clean_df[col] = clean_df[col].fillna('Unknown')

# Handle satisfaction missing values using median
clean_df['patient_satisfaction_score'] = clean_df[
    'patient_satisfaction_score'
].fillna(clean_df['patient_satisfaction_score'].median())

# Validate numeric fields
clean_df['patient_age'] = pd.to_numeric(clean_df['patient_age'], errors='coerce')
clean_df['patient_waittime'] = pd.to_numeric(clean_df['patient_waittime'], errors='coerce')

# Remove invalid records
clean_df = clean_df.dropna(subset=['patient_age', 'patient_waittime'])
clean_df = clean_df[
    (clean_df['patient_age'] >= 0) &
    (clean_df['patient_age'] <= 120) &
    (clean_df['patient_waittime'] >= 0)
]

# Remove exact duplicate rows
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

# Derived analytical features
clean_df['age_group'] = pd.cut(
    clean_df['patient_age'],
    bins=[-1, 17, 35, 50, 65, 120],
    labels=['Child', 'Young Adult', 'Adult', 'Senior Adult', 'Elderly']
)

clean_df['wait_time_category'] = pd.cut(
    clean_df['patient_waittime'],
    bins=[-1, 15, 30, 60, np.inf],
    labels=['Short (0-15 min)', 'Moderate (16-30 min)',
            'Long (31-60 min)', 'Very Long (60+ min)']
)

clean_df['satisfaction_category'] = pd.cut(
    clean_df['patient_satisfaction_score'],
    bins=[-np.inf, 2, 3, 4, 5],
    labels=['Low', 'Average', 'Good', 'Excellent']
)

print('Cleaned dataset shape:', clean_df.shape)
display(clean_df.head())


In [ ]:
# 5. Validation after cleaning
print('Missing values after cleaning:')
display(clean_df.isnull().sum().to_frame('Missing Count'))

print('Duplicate rows after cleaning:', clean_df.duplicated().sum())

print('Age range:', clean_df['patient_age'].min(), 'to', clean_df['patient_age'].max())
print('Wait-time range:', clean_df['patient_waittime'].min(), 'to', clean_df['patient_waittime'].max())
print('Satisfaction range:',
      clean_df['patient_satisfaction_score'].min(), 'to',
      clean_df['patient_satisfaction_score'].max())


In [ ]:
# 6. Key hospital KPIs
kpis = pd.DataFrame({
    'Metric': [
        'Total patient records',
        'Average patient wait time (minutes)',
        'Median patient wait time (minutes)',
        'Maximum patient wait time (minutes)',
        'Average satisfaction score',
        'Admission rate (%)'
    ],
    'Value': [
        len(clean_df),
        round(clean_df['patient_waittime'].mean(), 2),
        round(clean_df['patient_waittime'].median(), 2),
        clean_df['patient_waittime'].max(),
        round(clean_df['patient_satisfaction_score'].mean(), 2),
        round((clean_df['patient_admission_flag'].eq('Admission').mean()) * 100, 2)
    ]
})
display(kpis)


In [ ]:
# 7. Waiting-time analysis
department_wait = (
    clean_df.groupby('department_referral', dropna=False)
    .agg(
        patient_count=('patient_id', 'count'),
        average_wait_time=('patient_waittime', 'mean'),
        median_wait_time=('patient_waittime', 'median'),
        maximum_wait_time=('patient_waittime', 'max')
    )
    .sort_values('average_wait_time', ascending=False)
)

department_wait['average_wait_time'] = department_wait['average_wait_time'].round(2)
display(department_wait)


In [ ]:
# 8. Visualization: distribution of patient waiting time
plt.figure(figsize=(10, 5))
sns.histplot(clean_df['patient_waittime'], bins=30, kde=True)
plt.title('Distribution of Patient Waiting Time')
plt.xlabel('Waiting Time (minutes)')
plt.ylabel('Number of Patients')
plt.show()


In [ ]:
# 9. Visualization: average waiting time by department
plot_data = department_wait.reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_data, x='average_wait_time',
            y='department_referral')
plt.title('Average Waiting Time by Department')
plt.xlabel('Average Waiting Time (minutes)')
plt.ylabel('Department Referral')
plt.show()


In [ ]:
# 10. Visualization: admission status
plt.figure(figsize=(7, 5))
sns.countplot(data=clean_df, x='patient_admission_flag')
plt.title('Patient Admission Status')
plt.xlabel('Admission Status')
plt.ylabel('Number of Patients')
plt.xticks(rotation=20)
plt.show()


In [ ]:
# 11. Visualization: patient age groups
plt.figure(figsize=(9, 5))
sns.countplot(data=clean_df, x='age_group')
plt.title('Patient Distribution by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Number of Patients')
plt.xticks(rotation=20)
plt.show()


In [ ]:
# 12. Visualization: satisfaction score distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=clean_df, x='patient_satisfaction_score')
plt.title('Patient Satisfaction Score Distribution')
plt.xlabel('Satisfaction Score')
plt.ylabel('Number of Patients')
plt.show()


In [ ]:
# 13. Relationship between waiting time and satisfaction
wait_satisfaction = (
    clean_df.groupby('wait_time_category', observed=False)
    .agg(
        patient_count=('patient_id', 'count'),
        average_satisfaction=('patient_satisfaction_score', 'mean')
    )
    .reset_index()
)

wait_satisfaction['average_satisfaction'] = (
    wait_satisfaction['average_satisfaction'].round(2)
)
display(wait_satisfaction)

plt.figure(figsize=(9, 5))
sns.barplot(data=wait_satisfaction,
            x='wait_time_category',
            y='average_satisfaction')
plt.title('Average Satisfaction by Waiting-Time Category')
plt.xlabel('Waiting-Time Category')
plt.ylabel('Average Satisfaction Score')
plt.xticks(rotation=25)
plt.show()


In [ ]:
# 14. Admission analysis by department
admission_department = pd.crosstab(
    clean_df['department_referral'],
    clean_df['patient_admission_flag'],
    normalize='index'
).mul(100).round(2)

display(admission_department)

admission_department.plot(kind='bar', stacked=True, figsize=(11, 6))
plt.title('Admission Status Percentage by Department')
plt.xlabel('Department Referral')
plt.ylabel('Percentage of Patients')
plt.xticks(rotation=35, ha='right')
plt.legend(title='Admission Status')
plt.tight_layout()
plt.show()


In [ ]:
# 15. Patient demographics analytics
gender_summary = clean_df.groupby('patient_gender').agg(
    patient_count=('patient_id', 'count'),
    average_age=('patient_age', 'mean'),
    average_wait_time=('patient_waittime', 'mean'),
    average_satisfaction=('patient_satisfaction_score', 'mean')
).round(2)

display(gender_summary)

race_summary = clean_df['patient_race'].value_counts().to_frame('Patient Count')
display(race_summary)


In [ ]:
# 16. Dashboard-ready summary tables
dashboard_summary = {
    'department_waiting_time': department_wait.reset_index(),
    'admission_by_department': admission_department.reset_index(),
    'wait_time_vs_satisfaction': wait_satisfaction,
    'gender_summary': gender_summary.reset_index(),
    'race_summary': race_summary.reset_index()
}

for name, table in dashboard_summary.items():
    output_file = f'{name}.csv'
    table.to_csv(output_file, index=False)
    print('Saved:', output_file)


In [ ]:
# 17. Export cleaned data
OUTPUT_FILE = 'hospital_patient_flow_cleaned.csv'
clean_df.to_csv(OUTPUT_FILE, index=False)
print(f'Cleaned dataset exported to: {OUTPUT_FILE}')


## 18. Data Warehouse / Star Schema Proposal

A production hospital warehouse can use the following star schema:

### Fact tables
- **Fact_Patient_Visit:** patient, date, department, wait time, admission flag, satisfaction
- **Fact_Appointment:** appointment and operational events
- **Fact_Treatment:** treatment and cost information, when available
- **Fact_Lab_Result:** laboratory values, when available

### Dimension tables
- **Dim_Patient:** patient demographic attributes
- **Dim_Department:** department and specialization
- **Dim_Date:** date-based reporting attributes
- **Dim_Doctor:** doctor information, when available
- **Dim_Hospital_Location:** hospital branch or location, when available

The current dataset can populate patient-flow-related fields, while missing clinical and doctor-level sources would require additional source systems.


## 19. Ingestion, Monitoring, Security, and Privacy Proposal

### Ingestion
- Batch ingestion of CSV files for historical data.
- APIs or database connectors for hospital systems.
- Kafka or another streaming platform for real-time wearable and emergency events in a future implementation.

### Monitoring
- Track missing values and duplicates.
- Validate age, waiting time, and satisfaction ranges.
- Monitor ingestion failures and processing duration.
- Track storage and model performance when ML is introduced.

### Security and privacy
- Encrypt data in transit and at rest.
- Use role-based access control.
- Mask patient identifiers.
- Maintain audit logs.
- Apply retention and backup policies.

### Machine learning limitation
This dataset does not provide a clinically validated high-risk patient target. A risk classification model should only be developed after obtaining an appropriate target label and clinical approval. The current notebook therefore focuses on descriptive and operational analytics rather than claiming medical risk prediction.
